## 1. 数据分析与清洗

本 notebook 用于探索 SQL Generator No CoT 数据集的基本结构、数据质量、response 格式以及 SQL 复杂度。这里的分析结果会作为后续 `data_prepare.py` 数据处理 pipeline 的依据。

In [1]:
import pandas as pd

df = pd.read_csv(
    "hf://datasets/AI4DS/sql_generator_no_cot/training_no_cot_dataset.csv"
)

d:\miniconda\envs\d2l\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.1 数据集基本结构

先查看数据集的行数、列数、字段类型和 non-null 情况，确认数据是否成功导入，以及主要字段是什么。

In [2]:
# 查看数据的分析以及形状
print(df.shape)
# 查看数据信息
print(df.info())


(9399, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9399 entries, 0 to 9398
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   prompt    9399 non-null   object
 1   response  9399 non-null   object
dtypes: object(2)
memory usage: 147.0+ KB
None


**结果分析：**

原始数据集共有 **9399 行、2 列**，分别是 `prompt` 和 `response`。其中 `prompt` 是模型输入，`response` 是目标 SQL 输出。两列均为 non-null，说明没有明显缺失值。

**Report draft:**  
The original dataset contains 9,399 rows and two columns: `prompt` and `response`. The `prompt` column represents the model input, while the `response` column contains the target SQL query. Both columns are non-null, indicating that there are no obvious missing values at the basic structural level.

### 1.2 随机样本观察

随机抽取一条样本，人工观察 `prompt` 和 `response` 的实际内容，判断 `prompt` 是否只是用户问题本身。

In [3]:
sample =  df.sample(1) # 随机抽取一列查看propmt, response结果
print(sample["prompt"].iloc[0]) 
print(sample["response"].iloc[0])

You are a data science expert.
Below, you are presented with a database schema and a question.
Your task is to read the schema, understand the question, and generate a valid SQLite query to answer the question.

This schema offers an in-depth description of the database's architecture, detailing tables, columns, primary keys, foreign keys, and any pertinent information regarding relationships or constraints. Special attention should be given to the examples listed beside each column, as they directly hint at which columns are relevant to our query.


Database Schema
###
CREATE TABLE Customers
(
	ID INTEGER constraint Customers_pk primary key,
	age INTEGER, -- description: age of customers value description: � teenager: 13-19 years old. � elder: people aged over 65
);

###
Question: 
What is the total number of customers with an age below 30? 

Hint:
age below 30 refers to age < 30;

Please respond with a SQL query between ```sql and ``` that answers the question.

```sql
SELECT COUNT(I

**结果分析：**

随机样本显示，`prompt` 并不只是用户的自然语言问题，而是包含任务说明、数据库 schema、question，以及有时出现的 hint。因此，这里的 `prompt` 是完整模型输入模板。

**Report draft:**  
A randomly sampled prompt shows that the input is much longer than a normal user question. Each prompt usually contains task instructions, database schema information, the natural language question, and sometimes additional hints. Therefore, the prompt should be understood as the complete model input rather than the raw user query alone.

### 1.3 Prompt 和 Response 长度分析

由于 `prompt` 明显较长，需要分析输入和输出长度。这个结果会影响后续 tokenization 中 `max_input_length` 和 `max_target_length` 的设置。

In [4]:
df["prompt_length"] = df["prompt"].str.len()
df["response_length"] = df["response"].str.len()

df[["prompt_length", "response_length"]].describe()

,prompt_length,response_length
count,9399.000000,9399.000000
mean,1437.278753,183.490052
std,323.394642,75.468167
min,724.000000,36.000000
25%,1216.500000,137.000000
50%,1397.000000,175.000000
75%,1608.000000,222.000000
max,4758.000000,817.000000


In [5]:
# 看一下最短的几个prompt
df.sort_values("prompt_length").head(3)[["prompt", "response", "prompt_length"]]


,prompt,response,prompt_length
7244,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(TerritoryID) FROM Territo...,724
7012,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(TerritoryDescription) FRO...,751
6764,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(TerritoryID) FROM Territo...,759


In [6]:
# 看一下最长的几个prompt
df.sort_values("prompt_length").tail(3)[["prompt", "response", "prompt_length"]]

,prompt,response,prompt_length
7936,"You are a data science expert.\nBelow, you are...",```sql\nSELECT DISTINCT CASE WHEN SUM(CAST(str...,3994
420,"You are a data science expert.\nBelow, you are...",```sql\nSELECT SUM((T1.Sales / (1 - T1.Discoun...,4091
274,"You are a data science expert.\nBelow, you are...",```sql\nSELECT DISTINCT (T2.Sales / (1 - T2.di...,4758


**结果分析：**

最短的 prompt 大约为 **724-759** 个字符，最长的 prompt 大约达到 **3994-4758** 个字符，说明 prompt 长度差异明显。这是合理的，因为 prompt 包含 instruction、schema、question 和可能的 hint。

**Report draft:**  
The prompt length varies noticeably across the dataset. The shortest prompts are around 724-759 characters, while the longest prompts reach approximately 3994-4758 characters. This variation is expected because each prompt includes not only the natural language question, but also task instructions, database schema, and sometimes hints. This length variation should be considered when choosing the maximum input length for tokenization.

### 1.4 Response 长度分析

`response` 是模型需要生成的 SQL，因此也需要查看最短和最长 response，判断目标 SQL 的复杂程度。

In [7]:
# 看一下最短的 sql response
df.sort_values("response_length").head(3)[["prompt", "response", "response_length"]]

,prompt,response,response_length
7240,"You are a data science expert.\nBelow, you are...",```sql\nSELECT pages FROM langs ;\n```,36
8814,"You are a data science expert.\nBelow, you are...",```sql\nSELECT MAX(runtime) FROM movie ;\n```,43
2195,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(name) FROM person ;\n```,43


In [8]:
# 看一下最长的 sql response
df.sort_values("response_length").tail(3)[["prompt", "response", "response_length"]]

,prompt,response,response_length
669,"You are a data science expert.\nBelow, you are...",```sql\nSELECT (( SELECT T2.Value FROM Country...,741
1040,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(DISTINCT T1.CountryCode) ...,756
8725,"You are a data science expert.\nBelow, you are...",```sql\nSELECT CAST(SUM(CASE WHEN T1.age > 13 ...,817


**结果分析：**

最短的 response 大约 **36-43** 个字符，通常是简单 SQL；最长的 response 大约 **741-817** 个字符，可能包含 nested query、`CASE WHEN`、`JOIN`、聚合等复杂结构。

**Report draft:**  
The response length also varies across samples. The shortest responses are around 36-43 characters and usually correspond to simple SQL queries, while the longest responses are around 741-817 characters and may contain nested queries, joins, aggregations, or `CASE WHEN` expressions. This indicates that the dataset covers different levels of SQL complexity.

### 1.5 空字符串检查

虽然 `info()` 显示没有缺失值，但还需要检查空字符串或只包含空格、换行的文本，因为这些不会被 `isnull()` 识别。

In [9]:
empty_prompt = (df["prompt"].astype(str).str.strip() == "").sum()
empty_response = (df["response"].astype(str).str.strip() == "").sum()

empty_prompt, empty_response

(np.int64(0), np.int64(0))

**结果分析：**

`prompt` 和 `response` 中的空字符串数量均为 **0**，说明每条样本都有实际文本内容。

**Report draft:**  
In addition to checking missing values, we also checked for empty or whitespace-only strings. No empty entries were found in either the `prompt` or `response` column, which means every row contains valid text input and target output.

### 1.6 重复值检查与去重

重复样本可能导致模型反复看到相同训练样本，也可能在 train / validation / test split 时造成数据泄漏。因此需要在 split 之前检查并去重。

In [10]:
df.duplicated().sum()

np.int64(2)

In [11]:
df[df.duplicated(keep=False)].sort_values(["prompt", "response"]).head(10)

,prompt,response,prompt_length,response_length
3505,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(T1.film_id) FROM film AS ...,1170,140
4264,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(T1.film_id) FROM film AS ...,1170,140
5425,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(s_suppkey) FROM supplier ...,998,70
5720,"You are a data science expert.\nBelow, you are...",```sql\nSELECT COUNT(s_suppkey) FROM supplier ...,998,70


In [12]:
# 去重
df_clean = df.drop_duplicates().reset_index(drop=True)

df.shape, df_clean.shape

((9399, 4), (9397, 4))

**结果分析：**

数据集中发现 **2** 条重复副本。使用 `keep=False` 展示时会看到 4 行，因为它同时显示原始行和重复行。去重后生成 `df_clean`，保留原始 `df` 方便对比。

**Report draft:**  
Two duplicate rows were found in the dataset. When displaying all duplicated samples with `keep=False`, four rows are shown because both the original rows and their duplicated copies are included. These duplicate rows were removed before model training to reduce the risk of data leakage.

### 1.7 Response 格式检查与纯 SQL 提取

前面观察到 `response` 被 Markdown SQL code block 包裹，例如以 ```sql 开头并以 ``` 结尾。这里检查格式是否统一，并提取纯 SQL 到新列 `sql`。

In [13]:
starts_sql_block = df_clean["response"].astype(str).str.startswith("```sql").sum()
ends_code_block = df_clean["response"].astype(str).str.endswith("```").sum()

starts_sql_block, ends_code_block, len(df_clean)

(np.int64(9397), np.int64(9397), 9397)

In [14]:
df_clean["sql"] = (
    df_clean["response"]
    .str.replace("```sql", "", regex=False)
    .str.replace("```", "", regex=False)
    .str.strip()
)

df_clean[["response", "sql"]].head() # 看一下前几个有没有提纯成功

,response,sql
0,```sql\nSELECT COUNT(DISTINCT T2.teacher_accti...,SELECT COUNT(DISTINCT T2.teacher_acctid) FROM ...
1,```sql\nSELECT DISTINCT T2.LongName FROM Count...,SELECT DISTINCT T2.LongName FROM CountryNotes ...
2,```sql\nSELECT COUNT(DISTINCT T1.playerID) FRO...,SELECT COUNT(DISTINCT T1.playerID) FROM player...
3,```sql\nSELECT T2.FullName FROM Paper AS T1 IN...,SELECT T2.FullName FROM Paper AS T1 INNER JOIN...
4,```sql\nSELECT T2.StoreID FROM `Sales Orders` ...,SELECT T2.StoreID FROM `Sales Orders` AS T1 IN...


**结果分析：**

清洗后的 **9397** 条 response 全部以 ```sql 开头，并以 ``` 结尾，说明格式高度一致。因此可以安全地移除 code block 标记，并将纯 SQL 保存到 `sql` 列。原始 `response` 保留不变。

**Report draft:**  
All cleaned responses follow a consistent Markdown SQL code block format. Since all responses follow the same format, we extracted the pure SQL query into a new `sql` column by removing the surrounding code block markers. The original `response` column was kept unchanged for reference, while the cleaned `sql` column can be used for later training and evaluation.

### 1.8 SQL Query Type

检查每条 SQL 的第一个关键词，判断数据集主要包含哪类 SQL 操作。对于 text-to-SQL 查询生成任务，通常预期主要是 `SELECT` 查询。

In [15]:
df_clean["sql_start"] = df_clean["sql"].str.split().str[0].str.upper()

df_clean["sql_start"].value_counts()

sql_start
SELECT    9397
Name: count, dtype: int64

**结果分析：**

清洗后的 **9397** 条 SQL 全部以 `SELECT` 开头，说明该数据集关注的是信息检索类 SQL 查询生成，不涉及 `INSERT`、`UPDATE` 或 `DELETE` 等数据库修改操作。

**Report draft:**  
After extracting the first SQL keyword, all 9,397 cleaned SQL queries were found to start with `SELECT`. This indicates that the dataset focuses on SQL query generation for information retrieval, rather than database modification operations such as `INSERT`, `UPDATE`, or `DELETE`.

### 1.9 SQL 复杂度特征

为了粗略分析 SQL 复杂度，统计查询中是否包含 `JOIN`、`GROUP BY`、`ORDER BY`、`HAVING` 和 nested query。这些特征可以反映跨表连接、聚合、排序和子查询情况。

In [16]:
df_clean["has_join"] = df_clean["sql"].str.upper().str.contains("JOIN")
df_clean["has_group_by"] = df_clean["sql"].str.upper().str.contains("GROUP BY")
df_clean["has_order_by"] = df_clean["sql"].str.upper().str.contains("ORDER BY")
df_clean["has_having"] = df_clean["sql"].str.upper().str.contains("HAVING")
df_clean["has_nested_query"] = df_clean["sql"].str.upper().str.contains(r"\(\s*SELECT", regex=True)

df_clean[[
    "has_join",
    "has_group_by",
    "has_order_by",
    "has_having",
    "has_nested_query"
]].sum()

has_join            7202
has_group_by        1004
has_order_by        1701
has_having           141
has_nested_query     721
dtype: int64

**结果分析：**

在 **9397** 条清洗后的 SQL 中，**7202** 条包含 `JOIN`，说明大部分样本需要跨表查询。此外，**1004** 条包含 `GROUP BY`，**1701** 条包含 `ORDER BY`，**141** 条包含 `HAVING`，**721** 条包含 nested query。

这说明该数据集不只是简单单表查询，还包含大量跨表连接、聚合、排序和嵌套查询，对模型的 schema understanding 和 SQL generation 能力要求较高。

**Report draft:**  
We further analyzed SQL complexity by checking whether each query contains structures such as `JOIN`, `GROUP BY`, `ORDER BY`, `HAVING`, and nested subqueries. Among the 9,397 cleaned SQL queries, 7,202 contain `JOIN`, indicating that most examples require cross-table reasoning. In addition, 1,004 queries contain `GROUP BY`, 1,701 contain `ORDER BY`, 141 contain `HAVING`, and 721 contain nested subqueries. This suggests that the dataset includes a substantial number of complex SQL queries rather than only simple single-table selections.

### SQL Complexity Percentage

前面已经统计了不同 SQL 复杂结构出现的数量。为了让结果更直观，这里进一步计算每类结构在清洗后数据集中的占比。

这些百分比可以帮助我们判断数据集中复杂 SQL 的分布情况，例如有多少查询涉及跨表连接、排序、分组或嵌套查询。

In [17]:
complexity_counts = df_clean[[
    "has_join",
    "has_group_by",
    "has_order_by",
    "has_having",
    "has_nested_query"
]].sum()

complexity_percent = (complexity_counts / len(df_clean) * 100).round(2)

complexity_summary = pd.DataFrame({
    "count": complexity_counts,
    "percentage": complexity_percent
})

complexity_summary

,count,percentage
has_join,7202,76.64
has_group_by,1004,10.68
has_order_by,1701,18.10
has_having,141,1.50
has_nested_query,721,7.67


**结果分析：**

从统计结果可以看出，`JOIN` 的占比最高，说明数据集中大部分 SQL 都涉及跨表查询。这意味着模型不仅需要理解自然语言问题，还需要理解数据库 schema 中不同表之间的关系。

相比之下，`GROUP BY`、`ORDER BY` 和 nested query 的占比相对较低，但仍然覆盖了聚合、排序和子查询等常见复杂 SQL 结构。`HAVING` 的占比最低，说明条件聚合类查询在该数据集中较少。

**Report draft:**  
The percentage-based complexity summary shows that `JOIN` is the most frequent complex SQL structure, indicating that most queries require cross-table reasoning. Other structures such as `GROUP BY`, `ORDER BY`, and nested subqueries appear less frequently but still represent important SQL generation challenges. `HAVING` appears the least frequently, suggesting that conditional aggregation queries are relatively rare in this dataset.

## 2. Token Length Analysis for Preprocessing

前面统计的 `prompt_length` 和 `response_length` 是字符数量，而 T5-small 实际按照 token 处理文本。因此，在设置 `max_length` 之前，需要使用 T5-small tokenizer 统计 prompt 和 SQL 的 token 长度。

本节只用于分析 token 长度和评估截断风险，不会修改原始数据。特别需要关注 prompt 是否过长，因为 prompt 中包含数据库 schema 和 question，直接截断可能会丢失重要信息。

**Report draft:**  
Character length does not directly correspond to the number of tokens processed by T5-small. Therefore, tokenizer-based length analysis is required before selecting the maximum input and target lengths. This analysis helps estimate the proportion of examples that may be truncated during preprocessing.

In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Do not truncate here; the purpose is to measure the original token lengths.
prompt_token_ids = tokenizer(
    df_clean["prompt"].tolist(),
    truncation=False,
    add_special_tokens=True
)["input_ids"]

sql_token_ids = tokenizer(
    df_clean["sql"].tolist(),
    truncation=False,
    add_special_tokens=True
)["input_ids"]

df_clean["prompt_token_length"] = [len(tokens) for tokens in prompt_token_ids]
df_clean["sql_token_length"] = [len(tokens) for tokens in sql_token_ids]

df_clean[["prompt_token_length", "sql_token_length"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 1.0]
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (569 > 512). Running this sequence through the model will result in indexing errors


,prompt_token_length,sql_token_length
count,9397.000000,9397.000000
mean,371.642971,75.295094
std,98.506503,34.229745
min,157.000000,10.000000
50%,359.000000,71.000000
90%,494.000000,117.000000
95%,541.200000,135.000000
99%,686.040000,182.000000
100%,1313.000000,391.000000
max,1313.000000,391.000000


### Potential Truncation under Different Maximum Lengths

下面比较不同 `max_length` 下会被截断的 prompt 和 SQL 数量。根据这个结果，再决定 `data_prepare.py` 中的 tokenization 参数。

In [19]:
candidate_lengths = [512, 1024, 1536, 2048]

truncation_summary = pd.DataFrame({
    "max_length": candidate_lengths,
    "prompt_truncated_count": [
        (df_clean["prompt_token_length"] > length).sum()
        for length in candidate_lengths
    ],
    "sql_truncated_count": [
        (df_clean["sql_token_length"] > length).sum()
        for length in candidate_lengths
    ]
})

truncation_summary["prompt_truncated_percentage"] = (
    truncation_summary["prompt_truncated_count"] / len(df_clean) * 100
).round(2)

truncation_summary["sql_truncated_percentage"] = (
    truncation_summary["sql_truncated_count"] / len(df_clean) * 100
).round(2)

truncation_summary

,max_length,prompt_truncated_count,sql_truncated_count,prompt_truncated_percentage,sql_truncated_percentage
0,512,690,0,7.34,0.0
1,1024,4,0,0.04,0.0
2,1536,0,0,0.00,0.0
3,2048,0,0,0.00,0.0


## 2. 数据分析结论

通过以上分析，可以总结出当前数据集整体质量较好，适合进入后续 data preprocessing 阶段。原始数据集包含 **9399** 条样本和两列：`prompt` 与 `response`。其中 `prompt` 是完整模型输入，包含任务说明、数据库 schema、自然语言问题以及部分 hint；`response` 是目标 SQL 输出。

数据质量方面，`prompt` 和 `response` 均没有缺失值，也没有空字符串。数据集中存在少量完全重复样本，共 **2** 条重复副本，因此后续 preprocessing pipeline 中需要在划分 train / validation / test 之前执行去重，避免重复样本造成 data leakage。

格式方面，所有 `response` 都遵循统一的 Markdown SQL code block 格式，即以 ```sql 开头并以 ``` 结尾。因此可以稳定地提取纯 SQL，并保存到新的 `sql` 列中。后续训练和评估可以使用 `prompt` 作为输入，使用清洗后的 `sql` 作为目标输出。

长度和复杂度方面，prompt 长度差异明显，最长 prompt 超过 4000 个字符，说明后续 tokenization 时需要谨慎设置最大输入长度，避免重要 schema 信息被截断。SQL 复杂度分析显示，所有 SQL 都以 `SELECT` 开头，说明任务集中在查询生成；同时 **76.64%** 的 SQL 包含 `JOIN`，说明多数样本需要跨表推理。因此后续预处理应保留完整 prompt，而不是只保留自然语言 question。

**Report draft:**  
Overall, the dataset is suitable for downstream preprocessing and model training. The original dataset contains 9,399 prompt-response pairs, with no missing values or empty text entries. A small number of exact duplicates were found and should be removed before train-validation-test splitting to reduce the risk of data leakage. All responses follow a consistent Markdown SQL code block format, allowing the raw SQL query to be reliably extracted into a cleaned `sql` column. The prompt length varies substantially, with some prompts exceeding 4,000 characters, so input length limits should be considered during tokenization. SQL complexity analysis shows that all cleaned queries are `SELECT` statements, and 76.64% contain `JOIN`, indicating that most examples require schema-aware cross-table reasoning. Based on these findings, the preprocessing pipeline should remove duplicates, extract clean SQL targets, preserve the full prompt context, split the dataset reproducibly, and tokenize the prompt-SQL pairs for model training.
